# Generating Synthetic Multi-Turn Test Data

## Overview

Writing test cases manually is time-consuming and biased toward scenarios the author already knows about. Strands Evals [`ExperimentGenerator`](https://github.com/strands-agents/evals/blob/main/src/strands_evals/generators/experiment_generator.py) automates this by generating diverse test cases from a context description — including inputs, expected outputs, expected trajectories, and evaluation rubrics.

This notebook covers:
- **Section 7a**: Auto-generating test cases with `ExperimentGenerator.from_context_async()`, topic-based diversification with `TopicPlanner`, difficulty distribution, and auto-generated rubrics
- **Section 7c**: Scaling strategies — parallel async generation, saving/versioning experiments, and extending existing experiments

> **Section 7b** (DeepEval scenario generation) is covered by Ester in notebook 04.

## What You'll Learn

1. How to use `ExperimentGenerator.from_context_async()` to auto-generate test cases from tool/domain descriptions
2. How `TopicPlanner` distributes cases across diverse conversation themes via `num_topics`
3. How the generator automatically creates easy (30%), medium (50%), and hard (20%) test cases
4. How to auto-generate evaluation rubrics with `construct_evaluator_async()`
5. How to save experiments with `experiment.to_file()` and load them with `Experiment.from_file()`
6. How to extend existing experiments with `ExperimentGenerator.from_experiment_async()`

## Key Concepts

### ExperimentGenerator Methods

| Method | Input | Output | Use Case |
|--------|-------|--------|----------|
| `from_context_async()` | Context string + task description | Experiment with generated cases | Generate cases from tool descriptions and domain knowledge |
| `from_scratch_async()` | Topics list + task description | Experiment with generated cases | Generate cases from high-level topics |
| `from_experiment_async()` | Existing experiment + task description | New experiment inspired by source | Expand test suites while preserving style |
| `construct_evaluator_async()` | Context + evaluator class | Evaluator with generated rubric | Auto-generate evaluation criteria |

## 1. Setup

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()  # Required for async in Jupyter

from strands_evals import Case, Experiment
from strands_evals.generators import ExperimentGenerator
from strands_evals.evaluators import OutputEvaluator, TrajectoryEvaluator

print('Generator and evaluators imported.')

## 2. Section 7a — Auto-Generating Test Cases from Context

`ExperimentGenerator.from_context_async()` takes a context string describing your agent's capabilities and generates test cases that exercise those capabilities. The context should include:
- Available tools with their parameters and descriptions
- Domain knowledge the agent has access to
- Behavioral constraints and boundaries

The more detailed the context, the more relevant the generated test cases will be.

### Configuring the Generator

| Parameter | Purpose | Recommendation |
|-----------|---------|----------------|
| `input_type`, `output_type` | Types for Case fields | Use `str` for most conversational agents |
| `include_expected_output` | Generate expected outputs | `True` for output evaluation |
| `include_expected_trajectory` | Generate expected tool sequences | `True` for trajectory evaluation |
| `include_metadata` | Generate metadata (category, difficulty) | `True` for organizing test suites |
| `num_topics` | Distribute cases across N topics | 3-5 for diverse coverage |
| `max_parallel_num_cases` | Parallel generation workers | 5-10 for speed vs. cost balance |

In [ ]:
# Define the context — describe your agent's tools and domain
TRAVEL_AGENT_CONTEXT = """
Travel Booking Assistant with the following tools:

1. search_flights(origin: str, destination: str, departure_date: str, return_date: str = None)
   - Searches for available flights between two cities
   - Returns flight options with airline, times, price, and class

2. search_hotels(city: str, check_in: str, check_out: str, guests: int = 1)
   - Searches for available hotels in a city
   - Returns hotel options with star rating, price per night, and amenities

3. book_flight(flight_number: str, passenger_name: str, origin: str, destination: str, travel_date: str)
   - Books a specific flight for a passenger
   - Returns booking confirmation with reference number

4. book_hotel(hotel_name: str, guest_name: str, city: str, check_in: str, check_out: str)
   - Books a hotel room for a guest
   - Returns booking confirmation with reference number

5. get_bookings()
   - Retrieves all current bookings (flights and hotels)
   - Returns list of active and cancelled bookings

6. cancel_booking(booking_ref: str)
   - Cancels an existing booking by reference number
   - Returns cancellation confirmation

The agent helps users plan trips by searching for flights and hotels, making bookings,
reviewing existing reservations, and processing cancellations. It should confirm details
before booking and handle multi-step workflows (e.g., search → compare → book → verify).
"""

TASK_DESCRIPTION = (
    "A multi-turn travel booking assistant that helps users search for flights and hotels, "
    "make bookings, review reservations, and cancel bookings. Users may change their mind, "
    "ask follow-up questions, or request modifications across multiple conversation turns."
)

In [ ]:
# Generate test cases with topic diversification
generator = ExperimentGenerator(
    input_type=str,
    output_type=str,
    include_expected_output=True,
    include_expected_trajectory=True,
    include_metadata=True,
    max_parallel_num_cases=5,
)

print('Generating test cases with topic diversification...')
print('This uses TopicPlanner to distribute cases across 3 topics.')
print()

experiment = asyncio.get_event_loop().run_until_complete(
    generator.from_context_async(
        context=TRAVEL_AGENT_CONTEXT,
        task_description=TASK_DESCRIPTION,
        num_cases=6,       # Generate 6 test cases
        num_topics=3,      # Distribute across 3 topics for diversity
    )
)

print(f'Generated {len(experiment.cases)} test cases:')
for case in experiment.cases:
    traj = case.expected_trajectory or []
    meta = case.metadata or {}
    print(f'  {case.name}')
    print(f'    Input: {str(case.input)[:80]}...')
    print(f'    Expected trajectory: {traj}')
    print(f'    Metadata: {meta}')
    print()

### Difficulty Distribution

The generator automatically distributes test cases across difficulty levels:
- **Easy (30%)**: Straightforward single-tool tasks
- **Medium (50%)**: Multi-step workflows requiring context retention
- **Hard (20%)**: Complex scenarios with edge cases, pivots, or conflicting requirements

This distribution ensures your test suite covers both common paths and challenging edge cases.

### Auto-Generating Evaluation Rubrics

`construct_evaluator_async()` generates a task-specific rubric for any default evaluator (`OutputEvaluator`, `TrajectoryEvaluator`, `InteractionsEvaluator`). This saves you from writing rubrics manually while ensuring they're tailored to your domain.

In [ ]:
# Auto-generate a rubric for output evaluation
print('Generating evaluation rubric...')

evaluator = asyncio.get_event_loop().run_until_complete(
    generator.construct_evaluator_async(
        prompt=f'Context: {TRAVEL_AGENT_CONTEXT}\nTask: {TASK_DESCRIPTION}',
        evaluator=OutputEvaluator,
    )
)

print(f'Generated rubric:')
print(f'  {evaluator.rubric[:200]}...')

## 3. Section 7c — Scaling and Versioning

### Saving and Loading Experiments

Experiments can be serialized to JSON for version control, sharing, and reproducibility. This is essential for:
- Tracking how your test suite evolves over time
- Sharing test cases across team members
- Reproducing evaluation results from a specific point in time

In [ ]:
# Save the generated experiment
experiment.to_file('generated_travel_experiment.json')
print('Experiment saved to generated_travel_experiment.json')

# Load it back
loaded = Experiment.from_file('generated_travel_experiment.json')
print(f'Loaded experiment: {len(loaded.cases)} cases')
for case in loaded.cases:
    print(f'  {case.name}: {str(case.input)[:60]}...')

### Extending Existing Experiments

`from_experiment_async()` generates new test cases that are similar in style and structure to an existing experiment. This is useful for:
- Expanding coverage without starting from scratch
- Adding cases that complement existing ones (different difficulty, different topics)
- Iteratively growing your test suite as the agent's capabilities expand

In [ ]:
# Extend the experiment with 3 more cases
print('Extending experiment with 3 additional cases...')

extended = asyncio.get_event_loop().run_until_complete(
    generator.from_experiment_async(
        source_experiment=experiment,
        task_description=TASK_DESCRIPTION,
        num_cases=3,
        extra_information='Focus on edge cases: cancellation workflows, date changes, and multi-passenger bookings.',
    )
)

print(f'Extended experiment: {len(extended.cases)} cases (was {len(experiment.cases)})')
for case in extended.cases:
    print(f'  {case.name}: {str(case.input)[:70]}...')

In [ ]:
# Clean up generated files
import os
for f in ['generated_travel_experiment.json']:
    if os.path.exists(f):
        os.remove(f)
        print(f'Cleaned up {f}')

## Best Practices for Synthetic Data Generation

| Practice | Guidance |
|----------|----------|
| **Provide detailed context** | Include tool signatures, domain constraints, and behavioral boundaries. More context = more relevant test cases. |
| **Use `num_topics` for diversity** | 3-5 topics prevents the generator from clustering all cases around one scenario. |
| **Start small, then extend** | Generate 5-10 cases first, review them, then use `from_experiment_async()` to add more. |
| **Version your experiments** | Save with `to_file()` after each generation round. Track changes in version control. |
| **Review generated cases** | Auto-generated cases are a starting point. Review and edit them for accuracy and relevance. |
| **Mix generated and hand-written** | Use generated cases for broad coverage, hand-written cases for specific edge cases you know about. |

## Next Steps

- Continue to **`04-11-07-tool-simulation.ipynb`** for LLM-powered tool simulation with `ToolSimulator`